In [2]:
! uv add --active networkx pyarrow tabulate polars pandoc geopandas pyrosm duckdb
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Resolved 161 packages in 1ms
Checked 151 packages in 1ms


In [50]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

YEARS = range(2019, 2025)
AUDIT_DIR = PROJECT_ROOT / "outputs/maps/audit"
STATUSES = ["matched", "evaluated_unmatched", "not_evaluated"]

rows = []

for year in YEARS:
    path = AUDIT_DIR / f"global-match-status-{year}.parquet"

    if not path.is_file():
        raise FileNotFoundError(
            f"Audit absent pour {year} : {path}\n"
            f"Exécute : uv run python src/audit_global_matches.py {year}"
        )

    df = pd.read_parquet(path, columns=["accident_id", "status"])

    if df["accident_id"].isna().any() or df["accident_id"].duplicated().any():
        raise ValueError(f"{year} : accident_id manquant ou dupliqué")

    unknown = set(df["status"].dropna()) - set(STATUSES)
    if unknown or df["status"].isna().any():
        raise ValueError(f"{year} : statuts absents ou inconnus : {unknown}")

    counts = df["status"].value_counts()
    total = len(df)

    if total == 0:
        raise ValueError(f"{year} : audit vide")

    row = {
        "année": year,
        "total_BAAC": total,
        **{status: int(counts.get(status, 0)) for status in STATUSES},
    }

    if sum(row[status] for status in STATUSES) != total:
        raise ValueError(f"{year} : les statuts ne couvrent pas le total BAAC")

    row["matching_%_total"] = round(100 * row["matched"] / total, 2)
    row["matching_%_évalués"] = round(
        100 * row["matched"] / (row["matched"] + row["evaluated_unmatched"]),
        2,
    ) if row["matched"] + row["evaluated_unmatched"] else float("nan")
    row["non_évalués_%"] = round(100 * row["not_evaluated"] / total, 2)

    rows.append(row)

stats = pd.DataFrame(rows).set_index("année")
display(stats)
stats.to_csv()

,total_BAAC,matched,evaluated_unmatched,not_evaluated,matching_%_total,matching_%_évalués,non_évalués_%
année,,,,,,,
2019,58840,58501,190,149,99.42,99.68,0.25
2020,47744,47506,119,119,99.50,99.75,0.25
2021,56518,56428,90,0,99.84,99.84,0.00
2022,55302,55164,138,0,99.75,99.75,0.00
2023,54822,54681,137,4,99.74,99.75,0.01
2024,54402,54307,84,11,99.83,99.85,0.02


'année,total_BAAC,matched,evaluated_unmatched,not_evaluated,matching_%_total,matching_%_évalués,non_évalués_%\n2019,58840,58501,190,149,99.42,99.68,0.25\n2020,47744,47506,119,119,99.5,99.75,0.25\n2021,56518,56428,90,0,99.84,99.84,0.0\n2022,55302,55164,138,0,99.75,99.75,0.0\n2023,54822,54681,137,4,99.74,99.75,0.01\n2024,54402,54307,84,11,99.83,99.85,0.02\n'

In [2]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MATCHES_PATH = (
    PROJECT_ROOT
    / "outputs/maps/candidates-bretagne-2024.geoparquet"
)

assert MATCHES_PATH.is_file(), f"Fichier introuvable : {MATCHES_PATH}"

matches = gpd.read_parquet(MATCHES_PATH)

print(f"Lignes : {len(matches):,}")
print(f"CRS : {matches.crs}")
print("Colonnes :", matches.columns.tolist())
display(matches.head(10))

Lignes : 9,637
CRS : {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]},

,Num_Acc,geometry,osm_way_id,highway,distance_m,match_status,osm_source
0,202400000031.0,POINT (-1.6476 48.11337),4913513,secondary,2.986914,candidate,bretagne-240101.geoparquet
1,202400000031.0,POINT (-1.6476 48.11337),24771037,secondary,27.544033,candidate,bretagne-240101.geoparquet
2,202400000031.0,POINT (-1.6476 48.11337),467664834,secondary,27.544033,candidate,bretagne-240101.geoparquet
3,202400000037.0,POINT (-4.0519 47.99976),95316343,primary,5.440708,candidate,bretagne-240101.geoparquet
4,202400000037.0,POINT (-4.0519 47.99976),1123072435,primary,0.410681,candidate,bretagne-240101.geoparquet
5,202400000037.0,POINT (-4.0519 47.99976),590278141,primary,16.004179,candidate,bretagne-240101.geoparquet
6,202400000038.0,POINT (-4.0847 48.03158),112182608,trunk,0.078126,candidate,bretagne-240101.geoparquet
7,202400000038.0,POINT (-4.0847 48.03158),590278062,trunk,10.192384,candidate,bretagne-240101.geoparquet
8,202400000049.0,POINT (-1.84038 48.26549),25661477,tertiary,23.266869,candidate,bretagne-240101.geoparquet
9,202400000049.0,POINT (-1.84038 48.26549),1079019640,secondary,24.398914,candidate,bretagne-240101.geoparquet


In [16]:
accidents = matches[["Num_Acc"]].drop_duplicates()

sans_route = matches.loc[matches["osm_way_id"].isna()]


n_accidents = matches["Num_Acc"].nunique()
n_sans_route = matches.loc[
    matches["osm_way_id"].isna(), "Num_Acc"
].nunique()

print(f"Accidents retenus dans le test : {n_accidents:,}")
print(f"Accidents sans candidat : {n_sans_route:,}")
print(f"Accidents avec ≥ 1 candidat : {n_accidents - n_sans_route:,}")
print(f"Taux avec candidat : {(n_accidents - n_sans_route) / n_accidents:.1%}")

display(sans_route[["Num_Acc", "match_status", "distance_m", "geometry"]].head(1000))


Accidents retenus dans le test : 2,603
Accidents sans candidat : 272
Accidents avec ≥ 1 candidat : 2,331
Taux avec candidat : 89.6%


,Num_Acc,match_status,distance_m,geometry
46,202400000370.0,unmatched,NaN,POINT (-1.03789 47.40007)
90,202400000598.0,unmatched,NaN,POINT (-1.83107 47.62528)
231,202400001098.0,unmatched,NaN,POINT (-1.55997 48.82593)
232,202400001100.0,unmatched,NaN,POINT (-1.57033 48.81022)
264,202400001224.0,unmatched,NaN,POINT (-1.50841 48.63177)
...,...,...,...,...
9447,202400053447.0,unmatched,NaN,POINT (-1.15162 47.37639)
9521,202400053813.0,unmatched,NaN,POINT (-2.17047 47.48046)
9522,202400053820.0,unmatched,NaN,POINT (-1.2286 47.36406)
9588,202400054075.0,unmatched,NaN,POINT (-1.41141 47.7335)


In [20]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

lieux = pd.read_parquet(
    project_root / "BAAC/derived/2024/lieux-2024.parquet"
)

sans_route = (
    matches.loc[matches["osm_way_id"].isna()]
    .drop_duplicates(subset=["Num_Acc"])
    .copy()
)

sans_route["Num_Acc"] = sans_route["Num_Acc"].astype("string")
lieux["Num_Acc"] = lieux["Num_Acc"].astype("string")

colonnes_lieux = [
    "Num_Acc", "catr", "voie", "circ", "nbv",
    "vosp", "plan", "larrout", "infra", "situ", "vma",
]

details = sans_route.merge(
    lieux[colonnes_lieux],
    on="Num_Acc",
    how="left",
    validate="one_to_many",
    indicator=True,
)

print("Non-appariés :", len(sans_route))
print("Jointure avec lieux :")
display(details["_merge"].value_counts(dropna=False))

display(
    details[
        ["Num_Acc", "distance_m", "catr", "voie",
         "vma", "situ", "infra", "geometry"]
    ].head(30)
)

Non-appariés : 272
Jointure avec lieux :


_merge
both          318
left_only       0
right_only      0
Name: count, dtype: int64

,Num_Acc,distance_m,catr,voie,vma,situ,infra,geometry
0,202400000370.0,NaN,3.0,8,70,1,9,POINT (-1.03789 47.40007)
1,202400000598.0,NaN,3.0,125,80,1,5,POINT (-1.83107 47.62528)
2,202400000598.0,NaN,3.0,15,80,1,5,POINT (-1.83107 47.62528)
3,202400001098.0,NaN,3.0,AVRANCHES (ROUTE D'),80,1,0,POINT (-1.55997 48.82593)
4,202400001100.0,NaN,3.0,MATHURIN (RUE CHARLES),30,1,0,POINT (-1.57033 48.81022)
5,202400001224.0,NaN,5.0,N/A,30,1,0,POINT (-1.50841 48.63177)
6,202400001277.0,NaN,3.0,924,80,1,0,POINT (-1.4853 48.83691)
7,202400001327.0,NaN,3.0,D773,80,1,0,POINT (-2.08998 47.42321)
8,202400001585.0,NaN,3.0,D51,30,1,0,POINT (-2.4237 47.33084)
9,202400001589.0,NaN,3.0,16,80,1,0,POINT (-1.73247 47.40749)


In [21]:
details.to_csv("fails.csv")

In [22]:
import geopandas as gpd

roads = gpd.read_parquet(
    PROJECT_ROOT / "osm/derived/240101/bretagne-240101.geoparquet"
).to_crs("EPSG:2154")

fails = (
    matches.loc[matches["osm_way_id"].isna()]
    .drop_duplicates(subset=["Num_Acc"])
    .to_crs("EPSG:2154")
    .copy()
)

xmin, ymin, xmax, ymax = roads.total_bounds

study_extent = gpd.GeoSeries(
    [__import__("shapely").geometry.box(xmin, ymin, xmax, ymax)],
    crs="EPSG:2154",
)

fails["distance_to_osm_extent_m"] = fails.geometry.distance(
    study_extent.boundary.iloc[0]
)

display(
    fails["distance_to_osm_extent_m"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50])
    .to_frame("distance_to_osm_extent_m")
)

for threshold_m in [100, 500, 1_000, 5_000, 10_000]:
    n = (fails["distance_to_osm_extent_m"] <= threshold_m).sum()
    print(f"À ≤ {threshold_m:>6,} m de l'emprise OSM : {n:,} / {len(fails):,} ({n / len(fails):.1%})")

,distance_to_osm_extent_m
count,272.000000
mean,16096.141186
std,15212.696697
min,39.339376
1%,228.060012
5%,865.250923
10%,1979.450039
25%,4745.811586
50%,11672.358454
max,84794.783628


À ≤    100 m de l'emprise OSM : 2 / 272 (0.7%)
À ≤    500 m de l'emprise OSM : 5 / 272 (1.8%)
À ≤  1,000 m de l'emprise OSM : 16 / 272 (5.9%)
À ≤  5,000 m de l'emprise OSM : 71 / 272 (26.1%)
À ≤ 10,000 m de l'emprise OSM : 106 / 272 (39.0%)


In [24]:
import geopandas as gpd

caract = gpd.read_parquet(
    PROJECT_ROOT / "BAAC/derived/2024/caract-2024.geoparquet"
)

fails = (
    matches.loc[matches["osm_way_id"].isna(), ["Num_Acc", "geometry"]]
    .drop_duplicates(subset=["Num_Acc"])
    .copy()
)

fails["Num_Acc"] = fails["Num_Acc"].astype("string")
caract["Num_Acc"] = caract["Num_Acc"].astype("string")

fails_dep = fails.merge(
    caract[["Num_Acc", "dep", "com", "agg", "int"]],
    on="Num_Acc",
    how="left",
    validate="one_to_one",
)

print(
    fails_dep["dep"]
    .value_counts(dropna=False)
    .rename_axis("dep")
    .rename("accidents_sans_candidat")
    .to_frame()
)

     accidents_sans_candidat
dep                         
44                       124
50                       107
49                        13
53                         8
56                         6
35                         6
22                         4
29                         3
14                         1


In [25]:
departements_bretons = {"22", "29", "35", "56"}

ids_bretons = set(
    caract.loc[
        caract["dep"].astype("string").isin(departements_bretons),
        "Num_Acc",
    ].astype("string")
)

matches_bretagne = matches.loc[
    matches["Num_Acc"].astype("string").isin(ids_bretons)
]

n_total = matches_bretagne["Num_Acc"].nunique()
n_matched = matches_bretagne.loc[
    matches_bretagne["osm_way_id"].notna(), "Num_Acc"
].nunique()

print(f"Accidents bretons retenus : {n_total:,}")
print(f"Accidents bretons avec ≥ 1 candidat : {n_matched:,}")
print(f"Accidents bretons sans candidat : {n_total - n_matched:,}")
print(f"Taux breton sur les accidents retenus : {n_matched / n_total:.1%}")

Accidents bretons retenus : 2,340
Accidents bretons avec ≥ 1 candidat : 2,321
Accidents bretons sans candidat : 19
Taux breton sur les accidents retenus : 99.2%


In [28]:
matched = matches.loc[matches["osm_way_id"].notna()].copy()

nearest = (
    matched
    .sort_values(["Num_Acc", "distance_m", "osm_way_id"])
    .drop_duplicates(subset="Num_Acc", keep="first")
)

print("Accidents avec une voie attribuée :", nearest["Num_Acc"].nunique())
print(nearest[["Num_Acc", "osm_way_id", "highway", "distance_m"]].head(10))

Accidents avec une voie attribuée : 2331
           Num_Acc  osm_way_id    highway  distance_m
0   202400000031.0     4913513  secondary    2.986914
4   202400000037.0  1123072435    primary    0.410681
6   202400000038.0   112182608      trunk    0.078126
11  202400000049.0   369262913  secondary    8.793267
13  202400000053.0   534092734      trunk    0.706334
18  202400000061.0   111850247      trunk    0.182516
19  202400000070.0   148686818   tertiary    2.108640
21  202400000133.0   339766020  secondary    8.479749
23  202400000159.0   435278763  secondary    2.447044
24  202400000162.0    29419119   tertiary    1.162814


In [29]:
from pathlib import Path
import geopandas as gpd

root = PROJECT_ROOT

source = gpd.read_parquet(
    root / "outputs/maps/candidates-bretagne-2024.geoparquet"
)

n_total = source["Num_Acc"].nunique()
n_matched = source.loc[
    source["osm_way_id"].notna(), "Num_Acc"
].nunique()
n_unmatched = source.loc[
    source["osm_way_id"].isna(), "Num_Acc"
].nunique()

print(f"Accidents distincts : {n_total}")
print(f"Avec candidat : {n_matched}")
print(f"Sans candidat : {n_unmatched}")
print(f"Taux : {n_matched / n_total:.2%}")
print(f"Contrôle total : {n_matched + n_unmatched} / {n_total}")

Accidents distincts : 2603
Avec candidat : 2349
Sans candidat : 254
Taux : 90.24%
Contrôle total : 2603 / 2603


In [9]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

YEAR = 2024
BAAC_DIR = ROOT / "BAAC" / "derived" / str(YEAR)
MATCH_DIR = ROOT / "outputs" / "maps" / str(YEAR)


def normalize_num_acc(series):
    # Accepte les identifiants stockés comme 202400000001,
    # 202400000001.0 ou sous forme numérique dans les Parquet.
    values = pd.to_numeric(series, errors="coerce")
    invalid = values.isna() | values.ne(values.round())

    if invalid.any():
        print(f"Num_Acc ignorés (manquants/non entiers) : {invalid.sum():,}")
        display(series.loc[invalid].head(10))

    return values.astype("Int64").astype("string")


def read_baac(name, columns):
    path = BAAC_DIR / f"{name}-{YEAR}.parquet"
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path, columns=columns)
    df["Num_Acc"] = normalize_num_acc(df["Num_Acc"])
    return df


match_files = sorted(MATCH_DIR.glob("candidates-*.geoparquet"))
if not match_files:
    raise FileNotFoundError(f"Aucun match trouvé dans {MATCH_DIR}")

matches = pd.concat(
    [
        pd.read_parquet(
            path,
            columns=[
                "Num_Acc", "osm_way_id", "highway",
                "distance_m", "match_status", "osm_source",
            ],
        )
        for path in match_files
    ],
    ignore_index=True,
)
matches["Num_Acc"] = normalize_num_acc(matches["Num_Acc"])

# Si un accident est présent dans plusieurs extraits, retenir la voie
# à distance minimale. Conserver un unmatched si aucun match n'existe.
matches["_matched"] = matches["match_status"].eq("matched")
matches = (
    matches.sort_values(
        ["Num_Acc", "_matched", "distance_m", "osm_way_id"],
        ascending=[True, False, True, True],
        na_position="last",
        kind="stable",
    )
    .drop_duplicates("Num_Acc")
    .drop(columns="_matched")
    .reset_index(drop=True)
)
if not matches["Num_Acc"].is_unique:
    raise ValueError("Matches : Num_Acc non unique")

caract = read_baac(
    "caracteristiques",
    [
        "Num_Acc", "jour", "mois", "an", "hrmn",
        "dep", "com", "agg", "int", "lum", "atm", "col",
    ],
)
veh = read_baac(
    "vehicules", ["Num_Acc", "id_vehicule", "catv"]
)
usagers = read_baac(
    "usagers",
    ["Num_Acc", "id_vehicule", "id_usager", "catu", "grav"],
)

if not caract["Num_Acc"].is_unique:
    raise ValueError("Caractéristiques : Num_Acc non unique")

for name, df in [("caract", caract), ("veh", veh), ("usagers", usagers)]:
    common = matches["Num_Acc"].isin(df["Num_Acc"]).sum()
    print(f"Accidents présents dans {name} : {common:,} / {len(matches):,}")

if not matches["Num_Acc"].isin(veh["Num_Acc"]).any():
    raise ValueError("Aucun Num_Acc commun entre matches et véhicules")

for df in (veh, usagers):
    df["id_vehicule"] = df["id_vehicule"].astype("string").str.strip()

veh["catv"] = pd.to_numeric(veh["catv"], errors="coerce")
usagers["catu"] = pd.to_numeric(usagers["catu"], errors="coerce")
usagers["grav"] = pd.to_numeric(usagers["grav"], errors="coerce")

if veh.duplicated(["Num_Acc", "id_vehicule"]).any():
    raise ValueError("Véhicules : doublon sur (Num_Acc, id_vehicule)")

if usagers["id_usager"].notna().any() and usagers.duplicated(
    ["Num_Acc", "id_usager"]
).any():
    raise ValueError("Usagers : doublon sur (Num_Acc, id_usager)")

# Résumé des véhicules : une ligne par accident.
veh_summary = (
    veh.groupby("Num_Acc")
    .agg(
        n_vehicules=("id_vehicule", "size"),
        n_edpm=("catv", lambda s: s.eq(50).sum()),
        n_velos=("catv", lambda s: s.isin([1, 80]).sum()),
        categories_vehicules=(
            "catv",
            lambda s: tuple(sorted(s.dropna().astype(int).tolist())),
        ),
    )
    .reset_index()
)
veh_summary["has_edpm"] = veh_summary["n_edpm"].gt(0)
veh_summary["has_velo"] = veh_summary["n_velos"].gt(0)
veh_summary["avec_autre_vehicule_edpm"] = (
    veh_summary["has_edpm"]
    & veh_summary["n_vehicules"].gt(veh_summary["n_edpm"])
)
veh_summary["avec_vl_edpm"] = (
    veh_summary["has_edpm"]
    & veh_summary["Num_Acc"].isin(
        veh.loc[veh["catv"].eq(7), "Num_Acc"]
    )
)

# Jointure usagers-véhicules : la gravité est celle des usagers du mode,
# pas celle d'un piéton rattaché au véhicule qui l'a heurté.
mode_users = usagers.merge(
    veh,
    on=["Num_Acc", "id_vehicule"],
    how="inner",
    validate="many_to_one",
)
mode_users = mode_users.loc[
    mode_users["catu"].ne(3)
    & mode_users["catv"].isin([50, 1, 80])
].copy()
mode_users["mode"] = mode_users["catv"].map(
    {50: "edpm", 1: "velo", 80: "velo"}
)

# BAAC : 1 = indemne, 4 = blessé léger, 3 = hospitalisé, 2 = tué.
severity_rank = {1: 0, 4: 1, 3: 2, 2: 3}
mode_users["rang_gravite"] = mode_users["grav"].map(severity_rank)

gravite = (
    mode_users.groupby(["Num_Acc", "mode"])
    .agg(
        n_usagers=("id_usager", "size"),
        n_gravite_inconnue=("rang_gravite", lambda s: s.isna().sum()),
        rang_gravite_max=("rang_gravite", "max"),
    )
    .reset_index()
)
gravite["gravite_max"] = gravite["rang_gravite_max"].map(
    {
        0: "indemne",
        1: "blessé léger",
        2: "blessé hospitalisé",
        3: "tué",
    }
)

gravite = gravite.pivot(
    index="Num_Acc",
    columns="mode",
    values=["n_usagers", "n_gravite_inconnue", "gravite_max"],
)
gravite.columns = [
    f"{variable}_{mode}" for variable, mode in gravite.columns
]
gravite = gravite.reset_index()

# Le pivot ne crée pas de colonne pour un mode totalement absent.
for mode in ("edpm", "velo"):
    for variable in ("n_usagers", "n_gravite_inconnue", "gravite_max"):
        column = f"{variable}_{mode}"
        if column not in gravite.columns:
            gravite[column] = pd.NA

analyse_2024 = (
    matches
    .merge(caract, on="Num_Acc", how="left", validate="one_to_one")
    .merge(veh_summary, on="Num_Acc", how="left", validate="one_to_one")
    .merge(gravite, on="Num_Acc", how="left", validate="one_to_one")
)

if len(analyse_2024) != len(matches) or not analyse_2024["Num_Acc"].is_unique:
    raise ValueError("La table finale n'est pas au grain d'un accident")

# Si une jointure se perd, on s'arrête au lieu de compter des NaN comme False.
if analyse_2024["has_edpm"].isna().any():
    raise ValueError(
        "Des accidents matchés ne sont pas présents dans VEHICULES"
    )

print(f"Accidents joints : {len(analyse_2024):,}")
print(f"Accidents EDPM : {int(analyse_2024['has_edpm'].sum()):,}")
print(f"Accidents vélo (bicyclette + VAE) : {int(analyse_2024['has_velo'].sum()):,}")

display(
    analyse_2024.loc[
        analyse_2024["has_edpm"].eq(True),
        [
            "Num_Acc", "highway", "distance_m", "agg", "int",
            "n_vehicules", "avec_autre_vehicule_edpm",
            "avec_vl_edpm", "gravite_max_edpm",
        ],
    ].head(10)
)

Num_Acc ignorés (manquants/non entiers) : 1


54402   NaN
Name: Num_Acc, dtype: float64

Num_Acc ignorés (manquants/non entiers) : 1


125187   NaN
Name: Num_Acc, dtype: float64

Accidents présents dans caract : 54,391 / 54,391
Accidents présents dans veh : 54,391 / 54,391
Accidents présents dans usagers : 54,391 / 54,391
Accidents joints : 54,391
Accidents EDPM : 2,842
Accidents vélo (bicyclette + VAE) : 5,523


,Num_Acc,highway,distance_m,agg,int,n_vehicules,avec_autre_vehicule_edpm,avec_vl_edpm,gravite_max_edpm
15,202400000016,primary,0.090093,2.0,2.0,2,True,False,blessé léger
20,202400000021,primary,7.772789,2.0,2.0,1,False,False,indemne
25,202400000026,cycleway,5.578973,2.0,1.0,1,False,False,indemne
77,202400000078,secondary,1.149679,2.0,1.0,2,True,False,blessé léger
114,202400000115,residential,8.168579,2.0,1.0,1,False,False,tué
123,202400000124,residential,1.119328,2.0,3.0,2,True,True,blessé léger
129,202400000130,residential,1.145257,2.0,1.0,2,True,True,blessé léger
134,202400000135,secondary,2.303140,2.0,6.0,2,True,True,blessé léger
147,202400000148,residential,2.248143,2.0,3.0,1,False,False,indemne
180,202400000181,residential,8.139961,2.0,3.0,2,True,True,blessé hospitalisé


In [10]:
print("Matches :", matches["Num_Acc"].head(5).tolist())
print("Caract  :", caract["Num_Acc"].head(5).tolist())
print("Véhicules :", veh["Num_Acc"].head(5).tolist())
print("Usagers :", usagers["Num_Acc"].head(5).tolist())

print("\nTypes :")
for nom, df in [
    ("matches", matches),
    ("caract", caract),
    ("veh", veh),
    ("usagers", usagers),
]:
    print(nom, df["Num_Acc"].dtype, repr(df["Num_Acc"].iloc[0]))

print("\nRecouvrement des identifiants :")
ids_match = set(matches["Num_Acc"].dropna())
for nom, df in [("caract", caract), ("veh", veh), ("usagers", usagers)]:
    print(nom, len(ids_match.intersection(df["Num_Acc"].dropna())))

print("\nCatégories véhicule :")
print(veh["catv"].value_counts(dropna=False).head(15))
print("EDPM dans véhicules :", veh["catv"].eq(50).sum())
print("Vélos dans véhicules :", veh["catv"].isin([1, 80]).sum())

Matches : ['202400000001', '202400000002', '202400000003', '202400000004', '202400000005']
Caract  : ['202400000001', '202400000002', '202400000003', '202400000004', '202400000005']
Véhicules : ['202400000001', '202400000001', '202400000002', '202400000003', '202400000004']
Usagers : ['202400000001', '202400000001', '202400000002', '202400000002', '202400000003']

Types :
matches string '202400000001'
caract string '202400000001'
veh string '202400000001'
usagers string '202400000001'

Recouvrement des identifiants :
caract 54391
veh 54391
usagers 54391

Catégories véhicule :
catv
7     53911
33     7777
10     7062
1      5045
2      3090
50     2867
30     2036
32     1714
31     1609
15      936
34      789
80      781
37      708
14      652
43      559
Name: count, dtype: int64
EDPM dans véhicules : 2867
Vélos dans véhicules : 5826


In [15]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

YEAR = 2024
BAAC_DIR = ROOT / "BAAC" / "derived" / str(YEAR)
MATCH_DIR = ROOT / "outputs" / "maps" / str(YEAR)


def normalize_id(series):
    values = pd.to_numeric(series, errors="coerce")
    invalid = values.notna() & values.ne(values.round())
    if invalid.any():
        raise ValueError(f"Identifiants non entiers : {series.loc[invalid].head().tolist()}")
    return values.astype("Int64").astype("string")


def normalize_vehicle_id(series):
    return (
        series.astype("string")
        .str.replace(r"\s+", "", regex=True)
        .str.replace(r"\.0$", "", regex=True)
    )

def read_baac(name, columns):
    path = BAAC_DIR / f"{name}-{YEAR}.parquet"
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path, columns=columns)
    df["Num_Acc"] = normalize_id(df["Num_Acc"])
    return df.loc[df["Num_Acc"].notna()].copy()


def wide_blocks(df, prefix, fields):
    """Une ligne par Num_Acc ; autant de blocs numérotés que nécessaire."""
    if df.empty:
        return pd.DataFrame(columns=["Num_Acc"])

    df = df.copy()
    df["_position"] = df.groupby("Num_Acc").cumcount() + 1

    result = df.pivot(
        index="Num_Acc",
        columns="_position",
        values=fields,
    )
    result.columns = [
        f"{prefix}_{position}_{field}"
        for field, position in result.columns
    ]
    return result.reset_index()


match_files = sorted(MATCH_DIR.glob("candidates-*.geoparquet"))
if not match_files:
    raise FileNotFoundError(f"Aucun match trouvé dans {MATCH_DIR}")

matches = pd.concat(
    [
        pd.read_parquet(
            path,
            columns=[
                "Num_Acc", "osm_way_id", "highway",
                "distance_m", "match_status", "osm_source",
            ],
        )
        for path in match_files
    ],
    ignore_index=True,
)
matches["Num_Acc"] = normalize_id(matches["Num_Acc"])

if matches["Num_Acc"].isna().any():
    raise ValueError("Match sans Num_Acc")

matches["_matched"] = matches["match_status"].eq("matched")
matches = (
    matches.sort_values(
        ["Num_Acc", "_matched", "distance_m", "osm_way_id"],
        ascending=[True, False, True, True],
        na_position="last",
        kind="stable",
    )
    .drop_duplicates("Num_Acc")
    .drop(columns="_matched")
    .reset_index(drop=True)
)
if not matches["Num_Acc"].is_unique:
    raise ValueError("Matches : Num_Acc non unique")

caract = read_baac(
    "caracteristiques",
    [
        "Num_Acc", "jour", "mois", "an", "hrmn", "dep", "com",
        "agg", "int", "lum", "atm", "col",
    ],
)
veh = read_baac(
    "vehicules",
    [
        "Num_Acc", "id_vehicule", "num_veh", "catv",
        "senc", "obs", "obsm", "choc", "manv", "motor", "occutc",
    ],
)
usagers = read_baac(
    "usagers",
    [
        "Num_Acc", "id_usager", "id_vehicule", "num_veh",
        "place", "catu", "grav", "sexe", "an_nais", "trajet",
        "secu1", "secu2", "secu3", "locp", "actp", "etatp",
    ],
)

if not caract["Num_Acc"].is_unique:
    raise ValueError("Caractéristiques : Num_Acc non unique")

for df in (veh, usagers):
    df["id_vehicule"] = normalize_vehicle_id(df["id_vehicule"])

veh["catv"] = pd.to_numeric(veh["catv"], errors="coerce")
usagers["grav"] = pd.to_numeric(usagers["grav"], errors="coerce")
usagers["catu"] = pd.to_numeric(usagers["catu"], errors="coerce")

if veh.duplicated(["Num_Acc", "id_vehicule"]).any():
    raise ValueError("Véhicules : clé (Num_Acc, id_vehicule) dupliquée")
if usagers["id_usager"].notna().any() and usagers.duplicated(
    ["Num_Acc", "id_usager"]
).any():
    raise ValueError("Usagers : clé (Num_Acc, id_usager) dupliquée")

# Tri stable : un même véhicule/usager occupe toujours le même bloc.
veh = veh.sort_values(
    ["Num_Acc", "id_vehicule", "num_veh"],
    kind="stable",
    na_position="last",
).reset_index(drop=True)
usagers = usagers.sort_values(
    ["Num_Acc", "id_vehicule", "id_usager"],
    kind="stable",
    na_position="last",
).reset_index(drop=True)

veh_fields = [
    "id_vehicule", "num_veh", "catv", "senc", "obs",
    "obsm", "choc", "manv", "motor", "occutc",
]
usager_fields = [
    "id_usager", "id_vehicule", "num_veh", "place", "catu",
    "grav", "sexe", "an_nais", "trajet", "secu1", "secu2",
    "secu3", "locp", "actp", "etatp",
]

veh_wide = wide_blocks(veh, "veh", veh_fields)
usagers_wide = wide_blocks(usagers, "usager", usager_fields)

# Indicateurs dérivés ; ils ne remplacent pas les colonnes détaillées.
veh_counts = (
    veh.groupby("Num_Acc")
    .agg(
        n_vehicules=("catv", "size"),
        n_edpm=("catv", lambda s: s.eq(50).sum()),
        n_velos=("catv", lambda s: s.isin([1, 80]).sum()),
        n_vl=("catv", lambda s: s.eq(7).sum()),
    )
    .reset_index()
)
veh_counts["has_edpm"] = veh_counts["n_edpm"].gt(0)
veh_counts["has_velo"] = veh_counts["n_velos"].gt(0)

usager_counts = (
    usagers.groupby("Num_Acc")
    .agg(n_usagers=("id_usager", "size"))
    .reset_index()
)

# On calcule la gravité EDPM à partir de la catégorie du véhicule
# auquel l'usager est rattaché ; les piétons (catu=3) sont exclus.
mode_users = usagers.merge(
    veh[["Num_Acc", "id_vehicule", "catv"]],
    on=["Num_Acc", "id_vehicule"],
    how="left",
    validate="many_to_one",
)
mode_users = mode_users.loc[
    mode_users["catu"].ne(3) & mode_users["catv"].eq(50)
].copy()
mode_users["_rang"] = mode_users["grav"].map(
    {1: 0, 4: 1, 3: 2, 2: 3}
)
edpm_grav = (
    mode_users.groupby("Num_Acc")["_rang"]
    .max()
    .map({0: "indemne", 1: "blessé léger",
          2: "blessé hospitalisé", 3: "tué"})
    .rename("gravite_max_edpm")
    .reset_index()
)

accidents_2024 = (
    matches
    .merge(caract, on="Num_Acc", how="left", validate="one_to_one")
    .merge(veh_counts, on="Num_Acc", how="left", validate="one_to_one")
    .merge(usager_counts, on="Num_Acc", how="left", validate="one_to_one")
    .merge(edpm_grav, on="Num_Acc", how="left", validate="one_to_one")
    .merge(veh_wide, on="Num_Acc", how="left", validate="one_to_one")
    .merge(usagers_wide, on="Num_Acc", how="left", validate="one_to_one")
)

if len(accidents_2024) != len(matches) or not accidents_2024["Num_Acc"].is_unique:
    raise ValueError("La table finale n'a pas une ligne par accident")

if accidents_2024["n_vehicules"].isna().any():
    raise ValueError("Des accidents matchés sont absents de VEHICULES")

print(
    f"{len(accidents_2024):,} accidents ; "
    f"{len(accidents_2024.columns):,} colonnes ; "
    f"{int(accidents_2024['has_edpm'].sum()):,} accidents EDPM"
)
print(
    accidents_2024.loc[
        accidents_2024["has_edpm"].eq(True),
        [
            "Num_Acc", "highway", "agg", "int",
            "n_vehicules", "gravite_max_edpm",
            "veh_1_catv", "veh_2_catv",
        ],
    ].head(10)
)

54,391 accidents ; 975 colonnes ; 2,842 accidents EDPM
          Num_Acc      highway  agg  int  n_vehicules    gravite_max_edpm  \
15   202400000016      primary  2.0  2.0            2        blessé léger   
20   202400000021      primary  2.0  2.0            1             indemne   
25   202400000026     cycleway  2.0  1.0            1             indemne   
77   202400000078    secondary  2.0  1.0            2        blessé léger   
114  202400000115  residential  2.0  1.0            1                 tué   
123  202400000124  residential  2.0  3.0            2        blessé léger   
129  202400000130  residential  2.0  1.0            2        blessé léger   
134  202400000135    secondary  2.0  6.0            2        blessé léger   
147  202400000148  residential  2.0  3.0            1             indemne   
180  202400000181  residential  2.0  3.0            2  blessé hospitalisé   

    veh_1_catv veh_2_catv  
15          50         10  
20          50        NaN  
25          5

In [16]:
print("Maximum de véhicules par accident :", int(accidents_2024["n_vehicules"].max()))
print("Maximum d'usagers par accident :", int(accidents_2024["n_usagers"].max()))

print("\nColonnes par famille :")
for prefix in ("veh_", "usager_"):
    cols = [c for c in accidents_2024 if c.startswith(prefix)]
    print(prefix, len(cols))

print("\nColonnes renseignées pour 10 accidents ou moins :")
rare = accidents_2024.notna().sum().sort_values()
display(rare[rare <= 10].head(30))

Maximum de véhicules par accident : 11
Maximum d'usagers par accident : 56

Colonnes par famille :
veh_ 110
usager_ 840

Colonnes renseignées pour 10 accidents ou moins :


usager_56_etatp        1
usager_56_catu         1
usager_56_id_usager    1
usager_55_id_usager    1
usager_54_id_usager    1
usager_53_id_usager    1
usager_52_id_usager    1
usager_52_grav         1
usager_53_grav         1
usager_54_grav         1
usager_55_grav         1
usager_56_grav         1
usager_56_secu1        1
usager_55_secu1        1
usager_53_secu1        1
usager_55_etatp        1
usager_52_locp         1
usager_53_locp         1
usager_54_locp         1
usager_55_locp         1
usager_56_locp         1
usager_52_sexe         1
usager_53_sexe         1
usager_54_sexe         1
usager_55_sexe         1
usager_56_sexe         1
usager_55_catu         1
usager_52_secu1        1
usager_54_catu         1
usager_52_catu         1
dtype: int64

In [18]:
ids_56 = accidents_2024.loc[
    accidents_2024["n_usagers"].eq(56), "Num_Acc"
]

print("Accident(s) avec 56 usagers :", ids_56.tolist())

print(
    veh.loc[
        veh["Num_Acc"].isin(ids_56),
        ["Num_Acc", "id_vehicule", "num_veh", "catv", "occutc"],
    ]
)

print(
    usagers.loc[
        usagers["Num_Acc"].isin(ids_56),
        ["Num_Acc", "id_vehicule", "id_usager", "catu", "grav"],
    ]
    .groupby(["Num_Acc", "id_vehicule", "catu", "grav"], dropna=False)
    .size()
    .rename("n_usagers")
    .reset_index()
)

Accident(s) avec 56 usagers : ['202400034184']
            Num_Acc id_vehicule num_veh  catv occutc
58307  202400034184   155721557     A01    38     56
        Num_Acc id_vehicule  catu  grav  n_usagers
0  202400034184   155721557   1.0   1.0          1
1  202400034184   155721557   2.0   1.0         19
2  202400034184   155721557   2.0   3.0          3
3  202400034184   155721557   2.0   4.0         33


In [22]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TABLE_DIR = ROOT / "outputs" / "tables"
YEARS = range(2019, 2025)

# Les codes sont lus dans le schéma du premier fichier : le script
# de fabrication doit produire les mêmes colonnes chaque année.
first = TABLE_DIR / f"accidents-analyse-{YEARS.start}.geoparquet"
if not first.is_file():
    raise FileNotFoundError(first)

import pyarrow.parquet as pq

catv_cols = sorted(
    col for col in pq.read_schema(first).names
    if col.startswith("n_catv_") and col != "n_catv_inconnu"
)

rows = []

for year in YEARS:
    path = TABLE_DIR / f"accidents-analyse-{year}.geoparquet"
    if not path.is_file():
        raise FileNotFoundError(path)

    schema_cols = set(pq.read_schema(path).names)
    missing = set(["Num_Acc", *catv_cols]) - schema_cols
    if missing:
        raise ValueError(f"{year} : colonnes manquantes : {sorted(missing)}")

    df = pd.read_parquet(path, columns=["Num_Acc", *catv_cols])

    if df["Num_Acc"].isna().any() or not df["Num_Acc"].is_unique:
        raise ValueError(f"{year} : Num_Acc manquant ou dupliqué")

    total = len(df)
    if total == 0:
        raise ValueError(f"{year} : table vide")

    row = {"année": year, "total_accidents": total}

    for col in catv_cols:
        code = col.removeprefix("n_catv_")
        n = int(df[col].gt(0).sum())
        row[f"accidents_catv_{code}"] = n
        row[f"part_catv_{code}_pct"] = 100 * n / total

    rows.append(row)

par_annee = pd.DataFrame(rows).set_index("année")

# Tableau principal identique à celui du rapport.
table_edpm = par_annee[
    ["total_accidents", "accidents_catv_50", "part_catv_50_pct"]
].rename(columns={
    "total_accidents": "Accidents dans la table",
    "accidents_catv_50": "Accidents avec EDPM",
    "part_catv_50_pct": "Part des accidents avec EDPM (%)",
})
display(table_edpm.round(2))

# Différence de part = points de pourcentage, pas variation relative.
variations = pd.DataFrame(index=par_annee.index)
variations["Écart total vs année précédente"] = (
    par_annee["total_accidents"].diff()
)
variations["Variation total (%)"] = (
    100 * par_annee["total_accidents"].pct_change()
)
variations["Écart EDPM vs année précédente"] = (
    par_annee["accidents_catv_50"].diff()
)
variations["Variation accidents EDPM (%)"] = (
    100 * par_annee["accidents_catv_50"].pct_change()
)
variations["Écart part EDPM (points)"] = (
    par_annee["part_catv_50_pct"].diff()
)
display(variations.round(2))

# Comparaison des parts d'accidents impliquant chaque catégorie :
# sélectionne uniquement les catégories observées au moins une fois.
part_cols = [
    col for col in par_annee
    if col.startswith("part_catv_")
    and par_annee[col].gt(0).any()
]
parts_par_categorie = par_annee[part_cols].rename(
    columns=lambda col: col.removeprefix("part_catv_").removesuffix("_pct")
)
parts_par_categorie.columns.name = "code catv"
display(parts_par_categorie.round(2))

,Accidents dans la table,Accidents avec EDPM,Part des accidents avec EDPM (%)
année,,,
2019,58691,632,1.08
2020,47625,879,1.85
2021,56518,1496,2.65
2022,55302,2076,3.75
2023,54818,2373,4.33
2024,54391,2842,5.23


,Écart total vs année précédente,Variation total (%),Écart EDPM vs année précédente,Variation accidents EDPM (%),Écart part EDPM (points)
année,,,,,
2019,NaN,NaN,NaN,NaN,NaN
2020,-11066.0,-18.85,247.0,39.08,0.77
2021,8893.0,18.67,617.0,70.19,0.80
2022,-1216.0,-2.15,580.0,38.77,1.11
2023,-484.0,-0.88,297.0,14.31,0.57
2024,-427.0,-0.78,469.0,19.76,0.90


code catv,00,01,02,03,07,10,13,14,15,16,17,20,21,30,31,32,33,34,35,36,37,38,39,40,41,42,43,50,60,80,99
année,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019,0.52,8.21,6.16,1.26,76.29,10.09,0.68,1.26,1.57,0.07,1.32,0.26,0.42,4.99,3.19,4.01,13.61,2.22,0.04,0.25,1.21,0.29,0.05,0.20,0.02,0.06,0.78,1.08,0.24,0.43,0.70
2020,0.48,10.06,6.59,0.83,74.68,10.36,0.62,1.20,1.57,0.07,1.17,0.29,0.45,5.44,3.18,3.96,12.99,1.97,0.06,0.33,1.06,0.22,0.04,0.22,0.04,0.06,0.64,1.85,0.37,0.68,0.88
2021,0.39,9.45,6.65,0.88,75.51,11.23,0.68,1.27,1.63,0.06,1.05,0.16,0.47,5.83,2.96,3.89,12.91,1.58,0.03,0.31,1.09,0.21,0.07,0.24,0.02,0.01,1.08,2.65,0.42,0.93,0.73
2022,0.49,9.28,5.86,0.88,75.44,11.47,0.68,1.29,1.58,0.05,1.05,0.13,0.42,4.85,2.88,3.80,12.95,1.69,0.04,0.31,1.14,0.32,0.05,0.22,0.03,0.03,1.10,3.75,0.36,1.21,0.71
2023,0.47,8.93,5.91,0.86,75.48,11.73,0.67,1.29,1.63,0.08,0.94,0.08,0.46,3.97,3.19,3.26,13.64,1.62,0.06,0.32,1.21,0.29,0.07,0.20,0.04,0.07,1.12,4.33,0.46,1.37,0.69
2024,0.52,8.80,5.58,0.85,74.85,12.17,0.69,1.17,1.65,0.05,0.87,0.09,0.49,3.67,2.94,3.12,13.92,1.45,0.02,0.30,1.28,0.44,0.06,0.23,0.03,0.03,1.02,5.23,0.46,1.41,0.53


In [28]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TABLE_DIR = ROOT / "outputs" / "tables"
YEARS = range(2019, 2025)

frames = []

for year in YEARS:
    path = TABLE_DIR / f"accidents-analyse-{year}.geoparquet"
    if not path.is_file():
        raise FileNotFoundError(path)

    df = pd.read_parquet(
        path,
        columns=["Num_Acc", "agg", "highway", "n_catv_01"],
    )

    if df["Num_Acc"].isna().any() or df["Num_Acc"].duplicated().any():
        raise ValueError(f"{year} : Num_Acc manquant ou dupliqué")

    df = df.loc[df["n_catv_01"].gt(0)].copy()
    df["annee"] = year
    frames.append(df)

edpm = pd.concat(frames, ignore_index=True)

agg_code = pd.to_numeric(edpm["agg"], errors="coerce")
edpm["zone"] = agg_code.map({
    1: "Hors agglomération",
    2: "En agglomération",
}).fillna("Zone inconnue")

edpm["type_voie"] = (
    edpm["highway"]
    .astype("string")
    .fillna("Non apparié")
)

# Une ligne = un accident : size() compte donc les accidents, pas les véhicules.
par_annee = (
    edpm.groupby(
        ["annee", "zone", "type_voie"],
        dropna=False,
    )
    .size()
    .rename("accidents")
    .reset_index()
)
par_annee["total_zone_annee"] = (
    par_annee.groupby(["annee", "zone"])["accidents"]
    .transform("sum")
)
par_annee["part_dans_zone_pct"] = (
    100 * par_annee["accidents"] / par_annee["total_zone_annee"]
).round(2)

display(
    par_annee.sort_values(
        ["annee", "zone", "accidents"],
        ascending=[True, True, False],
    )
)

# Vue cumulée : composition des accidents EDPM dans chaque zone,
# tous les millésimes 2019–2024 réunis.
cumule = (
    edpm.groupby(["zone", "type_voie"], dropna=False)
    .size()
    .rename("accidents")
    .reset_index()
)
cumule["total_zone_2019_2024"] = (
    cumule.groupby("zone")["accidents"].transform("sum")
)
cumule["part_dans_zone_pct"] = (
    100 * cumule["accidents"] / cumule["total_zone_2019_2024"]
).round(2)

print(
    cumule.sort_values(
        ["zone", "accidents"],
        ascending=[True, False],
    )
)

print(f"Accidents EDPM comptés : {len(edpm):,}")
print(f"Non appariés : {edpm['highway'].isna().sum():,}")
print(f"Zone inconnue : {(edpm['zone'] == 'Zone inconnue').sum():,}")

,annee,zone,type_voie,accidents,total_zone_annee,part_dans_zone_pct
5,2019,En agglomération,residential,1129,4033,27.99
6,2019,En agglomération,secondary,910,4033,22.56
4,2019,En agglomération,primary,707,4033,17.53
8,2019,En agglomération,tertiary,654,4033,16.22
11,2019,En agglomération,unclassified,252,4033,6.25
7,2019,En agglomération,service,164,4033,4.07
1,2019,En agglomération,cycleway,162,4033,4.02
2,2019,En agglomération,living_street,32,4033,0.79
9,2019,En agglomération,track,9,4033,0.22
0,2019,En agglomération,Non apparié,6,4033,0.15


                  zone      type_voie  accidents  total_zone_2019_2024  \
5     En agglomération    residential       6676                 24708   
6     En agglomération      secondary       5544                 24708   
4     En agglomération        primary       4617                 24708   
8     En agglomération       tertiary       4062                 24708   
11    En agglomération   unclassified       1248                 24708   
7     En agglomération        service       1110                 24708   
1     En agglomération       cycleway       1055                 24708   
2     En agglomération  living_street        273                 24708   
9     En agglomération          track         41                 24708   
10    En agglomération          trunk         39                 24708   
0     En agglomération    Non apparié         30                 24708   
3     En agglomération       motorway         13                 24708   
18  Hors agglomération      secondary 

In [29]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TABLE_DIR = ROOT / "outputs" / "tables"
YEARS = range(2019, 2025)

columns = [
    "Num_Acc", "agg", "highway", "n_catv_50",
    "n_edpm_tues", "n_edpm_hospitalises",
]

frames = []

for year in YEARS:
    path = TABLE_DIR / f"accidents-analyse-{year}.geoparquet"
    if not path.is_file():
        raise FileNotFoundError(path)

    df = pd.read_parquet(path, columns=columns)
    if df["Num_Acc"].isna().any() or df["Num_Acc"].duplicated().any():
        raise ValueError(f"{year} : Num_Acc manquant ou dupliqué")

    frames.append(df.loc[df["n_catv_50"].gt(0)].copy())

edpm = pd.concat(frames, ignore_index=True)

zone_code = pd.to_numeric(edpm["agg"], errors="coerce")
edpm["zone"] = zone_code.map({
    1: "Hors agglomération",
    2: "En agglomération",
}).fillna("Zone inconnue")
edpm["type_voie"] = edpm["highway"].astype("string").fillna("Non apparié")

edpm["accident_mortel_edpm"] = edpm["n_edpm_tues"].gt(0)
edpm["accident_hospitalise_edpm"] = edpm["n_edpm_hospitalises"].gt(0)
edpm["accident_grave_edpm"] = (
    edpm["accident_mortel_edpm"] | edpm["accident_hospitalise_edpm"]
)

table_gravite = (
    edpm.groupby(["zone", "type_voie"], dropna=False)
    .agg(
        accidents_edpm=("Num_Acc", "size"),
        avec_edpm_tue=("accident_mortel_edpm", "sum"),
        avec_edpm_hospitalise=("accident_hospitalise_edpm", "sum"),
        avec_edpm_tue_ou_hospitalise=("accident_grave_edpm", "sum"),
    )
    .reset_index()
)
table_gravite["part_grave_parmi_accidents_edpm_pct"] = (
    100
    * table_gravite["avec_edpm_tue_ou_hospitalise"]
    / table_gravite["accidents_edpm"]
).round(1)

display(
    table_gravite.sort_values(
        ["zone", "accidents_edpm"],
        ascending=[True, False],
    )
)

,zone,type_voie,accidents_edpm,avec_edpm_tue,avec_edpm_hospitalise,avec_edpm_tue_ou_hospitalise,part_grave_parmi_accidents_edpm_pct
5,En agglomération,residential,2873,31,416,447,15.6
6,En agglomération,secondary,2145,34,332,364,17.0
4,En agglomération,primary,1856,26,258,284,15.3
8,En agglomération,tertiary,1544,19,266,285,18.5
7,En agglomération,service,522,3,85,88,16.9
11,En agglomération,unclassified,445,4,89,93,20.9
1,En agglomération,cycleway,414,6,63,69,16.7
2,En agglomération,living_street,133,1,17,18,13.5
10,En agglomération,trunk,10,0,1,1,10.0
3,En agglomération,motorway,7,1,1,2,28.6


In [4]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Choix de l'analyse
YEARS = range(2019, 2020)
CATV = 1
LABEL = "VELO"

# Pour analyser les bicyclettes :
# CATV = 1
# LABEL = "VÉLO"

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ACC_DIR = ROOT / "outputs" / "tables"
BAAC_DIR = ROOT / "BAAC" / "derived"
CATV_COL = f"n_catv_{CATV:02d}"

accidents_par_an = []
gravite_par_an = []
toutes_catv = set()

for year in YEARS:
    acc_path = ACC_DIR / f"accidents-analyse-{year}.geoparquet"
    veh_path = BAAC_DIR / str(year) / f"vehicules-{year}.parquet"
    usa_path = BAAC_DIR / str(year) / f"usagers-{year}.parquet"

    for path in (acc_path, veh_path, usa_path):
        if not path.is_file():
            raise FileNotFoundError(path)

    colonnes_acc = pq.read_schema(acc_path).names
    colonnes_veh = pq.read_schema(veh_path).names
    colonnes_usa = pq.read_schema(usa_path).names

    requises = [
        (acc_path, colonnes_acc,
         {"Num_Acc", "agg", "highway", "n_vehicules",
          "n_pietons", CATV_COL}),
        (veh_path, colonnes_veh,
         {"Num_Acc", "id_vehicule", "catv"}),
        (usa_path, colonnes_usa,
         {"Num_Acc", "id_vehicule", "catu", "grav"}),
    ]
    for path, disponibles, necessaires in requises:
        manquantes = necessaires - set(disponibles)
        if manquantes:
            raise KeyError(f"{path.name} : colonnes absentes {sorted(manquantes)}")

    catv_cols = sorted(
        c for c in colonnes_acc
        if c.startswith("n_catv_") and c != "n_catv_inconnu"
    )
    toutes_catv.update(catv_cols)

    colonnes_lues = [
        "Num_Acc", "agg", "highway", "n_vehicules", "n_pietons",
        *catv_cols,
    ]
    if "n_catv_inconnu" in colonnes_acc:
        colonnes_lues.append("n_catv_inconnu")

    a = pd.read_parquet(acc_path, columns=colonnes_lues)
    if a["Num_Acc"].isna().any() or a["Num_Acc"].duplicated().any():
        raise ValueError(f"{year} : Num_Acc absent ou dupliqué")

    a = a.loc[pd.to_numeric(a[CATV_COL], errors="coerce").gt(0)].copy()
    a["Num_Acc"] = a["Num_Acc"].astype("string")
    if "n_catv_inconnu" not in a:
        a["n_catv_inconnu"] = 0
    a["annee"] = year
    accidents_par_an.append(a)

    v = pd.read_parquet(
        veh_path, columns=["Num_Acc", "id_vehicule", "catv"]
    )
    u = pd.read_parquet(
        usa_path, columns=["Num_Acc", "id_vehicule", "catu", "grav"]
    )

    for df in (v, u):
        df["Num_Acc"] = df["Num_Acc"].astype("string")
        df["id_vehicule"] = df["id_vehicule"].astype("string")

    v = v.loc[
        v["Num_Acc"].isin(a["Num_Acc"])
        & pd.to_numeric(v["catv"], errors="coerce").eq(CATV)
    ].copy()
    u = u.loc[u["Num_Acc"].isin(a["Num_Acc"])].copy()
    u = u.loc[
        pd.to_numeric(u["catu"], errors="coerce").isin([1, 2])
    ].copy()

    if v.duplicated(["Num_Acc", "id_vehicule"]).any():
        raise ValueError(f"{year} : véhicules de la catégorie dupliqués")

    cible = u.merge(
        v[["Num_Acc", "id_vehicule"]],
        on=["Num_Acc", "id_vehicule"],
        how="inner",
        validate="many_to_one",
    )

    grav = pd.to_numeric(cible["grav"], errors="coerce")
    for code, nom in [
        (1, "indemnes"),
        (4, "blesses_legers"),
        (3, "hospitalises"),
        (2, "tues"),
    ]:
        cible[nom] = grav.eq(code).astype(int)

    par_accident = (
        cible.groupby("Num_Acc", as_index=False)[
            ["indemnes", "blesses_legers", "hospitalises", "tues"]
        ]
        .sum()
    )
    par_accident["annee"] = year
    gravite_par_an.append(par_accident)

acc = pd.concat(accidents_par_an, ignore_index=True)

for col in toutes_catv:
    if col not in acc:
        acc[col] = 0

gravite_acc = pd.concat(gravite_par_an, ignore_index=True)
acc = acc.merge(
    gravite_acc,
    on=["annee", "Num_Acc"],
    how="left",
    validate="one_to_one",
)

etats = ["indemnes", "blesses_legers", "hospitalises", "tues"]
for etat in etats:
    acc[etat] = acc[etat].fillna(0).astype(int)
    acc[f"avec_{etat}"] = acc[etat].gt(0)

acc["zone"] = (
    pd.to_numeric(acc["agg"], errors="coerce")
    .map({1: "Hors agglomération", 2: "En agglomération"})
    .fillna("Zone inconnue")
)
acc["type_voie"] = acc["highway"].astype("string").fillna("Non apparié")

group_cols = ["zone", "type_voie"]
n_col = f"Accidents {LABEL}"
g = acc.groupby(group_cols, dropna=False)

table_gravite = g.size().rename(n_col).to_frame()

libelles = {
    "indemnes": "indemne",
    "blesses_legers": "blessé léger",
    "hospitalises": "hospitalisé",
    "tues": "tué",
}

for etat, libelle in libelles.items():
    col_accidents = f"Avec {LABEL} {libelle}"
    col_part = f"Part avec {LABEL} {libelle} (%)"
    col_personnes = f"{LABEL} {etat} (personnes)"
    table_gravite[col_accidents] = g[f"avec_{etat}"].sum()
    table_gravite[col_part] = (
        100 * table_gravite[col_accidents] / table_gravite[n_col]
    ).round(1)
    table_gravite[col_personnes] = g[etat].sum()

gravite_cols = table_gravite.columns.drop(n_col).tolist()

presence = acc[sorted(toutes_catv)].fillna(0).gt(0)
presence[group_cols] = acc[group_cols]
tiers = presence.groupby(group_cols, dropna=False).sum()
tiers = tiers.drop(columns=CATV_COL).rename(
    columns=lambda c: f"catv {c.removeprefix('n_catv_')}"
)

extras = acc[group_cols].copy()
extras["Sans autre véhicule"] = acc["n_vehicules"].eq(1)
extras[f"Autre {LABEL}"] = acc[CATV_COL].gt(1)
extras["Avec piéton"] = acc["n_pietons"].gt(0)
extras["Catégorie inconnue"] = acc["n_catv_inconnu"].gt(0)
extras = extras.groupby(group_cols, dropna=False).sum()

table_effectifs = table_gravite.join(extras).join(tiers).reset_index()
tiers_cols = [
    col for col in table_effectifs
    if col not in [*group_cols, n_col, *gravite_cols]
    and table_effectifs[col].sum() > 0
]

table = table_effectifs.copy()
for col in tiers_cols:
    table[col] = [
        f"{int(nb)} ({100 * nb / total:.1f} %)"
        for nb, total in zip(table[col], table[n_col])
    ]

table = (
    table[[*group_cols, n_col, *gravite_cols, *tiers_cols]]
    .sort_values(["zone", n_col], ascending=[True, False])
    .reset_index(drop=True)
)
display(table)

,zone,type_voie,Accidents VELO,Avec VELO indemne,Part avec VELO indemne (%),VELO indemnes (personnes),Avec VELO blessé léger,Part avec VELO blessé léger (%),VELO blesses_legers (personnes),Avec VELO hospitalisé,Part avec VELO hospitalisé (%),VELO hospitalises (personnes),Avec VELO tué,Part avec VELO tué (%),VELO tues (personnes),Sans autre véhicule,Autre VELO,Avec piéton,catv 00,catv 02,catv 03,catv 07,catv 10,catv 13,catv 14,catv 15,catv 16,catv 17,catv 20,catv 21,catv 30,catv 31,catv 32,catv 33,catv 34,catv 36,catv 37,catv 38,catv 39,catv 40,catv 43,catv 50,catv 60,catv 80,catv 99
0,En agglomération,residential,1129,79,7.0,79,837,74.1,850,205,18.2,206,12,1.1,12,184 (16.3 %),23 (2.0 %),90 (8.0 %),6 (0.5 %),8 (0.7 %),5 (0.4 %),713 (63.2 %),89 (7.9 %),5 (0.4 %),12 (1.1 %),1 (0.1 %),0 (0.0 %),0 (0.0 %),3 (0.3 %),1 (0.1 %),24 (2.1 %),4 (0.4 %),12 (1.1 %),6 (0.5 %),2 (0.2 %),1 (0.1 %),12 (1.1 %),4 (0.4 %),1 (0.1 %),7 (0.6 %),2 (0.2 %),4 (0.4 %),1 (0.1 %),1 (0.1 %),4 (0.4 %)
1,En agglomération,secondary,910,63,6.9,63,629,69.1,633,194,21.3,196,26,2.9,26,116 (12.7 %),14 (1.5 %),66 (7.3 %),2 (0.2 %),12 (1.3 %),12 (1.3 %),591 (64.9 %),65 (7.1 %),2 (0.2 %),7 (0.8 %),7 (0.8 %),0 (0.0 %),3 (0.3 %),3 (0.3 %),2 (0.2 %),8 (0.9 %),8 (0.9 %),11 (1.2 %),22 (2.4 %),6 (0.7 %),0 (0.0 %),11 (1.2 %),3 (0.3 %),1 (0.1 %),4 (0.4 %),2 (0.2 %),3 (0.3 %),0 (0.0 %),0 (0.0 %),4 (0.4 %)
2,En agglomération,primary,707,55,7.8,56,523,74.0,530,111,15.7,113,16,2.3,16,87 (12.3 %),24 (3.4 %),48 (6.8 %),1 (0.1 %),4 (0.6 %),2 (0.3 %),437 (61.8 %),51 (7.2 %),1 (0.1 %),13 (1.8 %),9 (1.3 %),2 (0.3 %),5 (0.7 %),1 (0.1 %),1 (0.1 %),12 (1.7 %),7 (1.0 %),14 (2.0 %),26 (3.7 %),5 (0.7 %),0 (0.0 %),9 (1.3 %),2 (0.3 %),0 (0.0 %),2 (0.3 %),3 (0.4 %),8 (1.1 %),0 (0.0 %),1 (0.1 %),0 (0.0 %)
3,En agglomération,tertiary,654,42,6.4,42,428,65.4,435,165,25.2,167,21,3.2,21,83 (12.7 %),12 (1.8 %),41 (6.3 %),5 (0.8 %),7 (1.1 %),8 (1.2 %),451 (69.0 %),46 (7.0 %),5 (0.8 %),2 (0.3 %),1 (0.2 %),0 (0.0 %),2 (0.3 %),1 (0.2 %),2 (0.3 %),9 (1.4 %),2 (0.3 %),4 (0.6 %),7 (1.1 %),1 (0.2 %),0 (0.0 %),6 (0.9 %),0 (0.0 %),2 (0.3 %),0 (0.0 %),0 (0.0 %),3 (0.5 %),0 (0.0 %),1 (0.2 %),1 (0.2 %)
4,En agglomération,unclassified,252,15,6.0,15,167,66.3,169,64,25.4,65,5,2.0,5,43 (17.1 %),6 (2.4 %),17 (6.7 %),0 (0.0 %),3 (1.2 %),1 (0.4 %),165 (65.5 %),19 (7.5 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),1 (0.4 %),0 (0.0 %),0 (0.0 %),2 (0.8 %),1 (0.4 %),3 (1.2 %),4 (1.6 %),0 (0.0 %),0 (0.0 %),1 (0.4 %),1 (0.4 %),0 (0.0 %),2 (0.8 %),1 (0.4 %),1 (0.4 %),0 (0.0 %),0 (0.0 %),0 (0.0 %)
5,En agglomération,service,164,12,7.3,12,125,76.2,126,27,16.5,27,3,1.8,3,21 (12.8 %),5 (3.0 %),13 (7.9 %),1 (0.6 %),3 (1.8 %),1 (0.6 %),111 (67.7 %),8 (4.9 %),1 (0.6 %),3 (1.8 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),1 (0.6 %),1 (0.6 %),1 (0.6 %),0 (0.0 %),0 (0.0 %),6 (3.7 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),2 (1.2 %),0 (0.0 %),0 (0.0 %),1 (0.6 %)
6,En agglomération,cycleway,162,14,8.6,14,134,82.7,137,18,11.1,18,1,0.6,1,22 (13.6 %),13 (8.0 %),14 (8.6 %),1 (0.6 %),1 (0.6 %),4 (2.5 %),91 (56.2 %),13 (8.0 %),1 (0.6 %),3 (1.9 %),1 (0.6 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),1 (0.6 %),0 (0.0 %),5 (3.1 %),0 (0.0 %),0 (0.0 %),1 (0.6 %),1 (0.6 %),0 (0.0 %),2 (1.2 %),0 (0.0 %),2 (1.2 %),0 (0.0 %),3 (1.9 %),0 (0.0 %)
7,En agglomération,living_street,32,6,18.8,6,18,56.2,19,7,21.9,7,0,0.0,0,11 (34.4 %),1 (3.1 %),7 (21.9 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),16 (50.0 %),1 (3.1 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),1 (3.1 %),0 (0.0 %),1 (3.1 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),1 (3.1 %),0 (0.0 %),0 (0.0 %),0 (0.0 %)
8,En agglomération,track,9,1,11.1,1,6,66.7,8,3,33.3,3,0,0.0,0,1 (11.1 %),2 (22.2 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),5 (55.6 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),1 (11.1 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %),0 (0.0 %

In [16]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# ── Choix de l'analyse ───────────────────────────────────────────────
YEARS = range(2019, 2025)
CATV = 2
LABEL = "CYCLO"

# Pour les EDPM :
#CATV = 50
#LABEL = "EDPM"
# ─────────────────────────────────────────────────────────────────────

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ACC_DIR = ROOT / "outputs" / "tables"
BAAC_DIR = ROOT / "BAAC" / "derived"
CATV_COL = f"n_catv_{CATV:02d}"

# Autres véhicules à propulsion non motorisée.
# Le VAE (80) est regroupé ici avec les cycles.
NON_MOTORISES = {1, 60, 80}

accidents_par_an = []
gravite_par_an = []
toutes_catv = set()


def normaliser_num_acc(serie):
    nombres = pd.to_numeric(serie, errors="raise")
    valeurs = nombres.dropna()

    if (valeurs % 1 != 0).any():
        raise ValueError("Num_Acc contient des valeurs non entières")

    if (valeurs.abs() >= 2**53).any():
        raise ValueError(
            "Num_Acc dépasse la précision entière garantie "
            "pour un float64"
        )

    return nombres.astype("Int64").astype("string")

for year in YEARS:
    acc_path = ACC_DIR / f"accidents-analyse-{year}.geoparquet"
    veh_path = BAAC_DIR / str(year) / f"vehicules-{year}.parquet"
    usa_path = BAAC_DIR / str(year) / f"usagers-{year}.parquet"

    for path in (acc_path, veh_path, usa_path):
        if not path.is_file():
            raise FileNotFoundError(path)

    colonnes_acc = pq.read_schema(acc_path).names
    colonnes_veh = pq.read_schema(veh_path).names
    colonnes_usa = pq.read_schema(usa_path).names

    requises = [
        (
            acc_path,
            colonnes_acc,
            {
                "Num_Acc", "agg", "highway", "n_vehicules",
                "n_pietons", CATV_COL,
            },
        ),
        (
            veh_path,
            colonnes_veh,
            {"Num_Acc", "id_vehicule", "catv"},
        ),
        (
            usa_path,
            colonnes_usa,
            {"Num_Acc", "id_vehicule", "catu", "grav"},
        ),
    ]

    for path, disponibles, necessaires in requises:
        manquantes = necessaires - set(disponibles)
        if manquantes:
            raise KeyError(
                f"{path.name} : colonnes absentes {sorted(manquantes)}"
            )

    catv_cols = sorted(
        col for col in colonnes_acc
        if col.startswith("n_catv_") and col != "n_catv_inconnu"
    )
    toutes_catv.update(catv_cols)

    colonnes_lues = [
        "Num_Acc", "agg", "highway", "n_vehicules",
        "n_pietons", *catv_cols,
    ]
    if "n_catv_inconnu" in colonnes_acc:
        colonnes_lues.append("n_catv_inconnu")

    a = pd.read_parquet(acc_path, columns=colonnes_lues)

    if a["Num_Acc"].isna().any() or a["Num_Acc"].duplicated().any():
        raise ValueError(f"{year} : Num_Acc absent ou dupliqué")

    a = a.loc[
        pd.to_numeric(a[CATV_COL], errors="coerce").gt(0)
    ].copy()
    a["Num_Acc"] = a["Num_Acc"].astype("string")

    if "n_catv_inconnu" not in a:
        a["n_catv_inconnu"] = 0

    a["annee"] = year
    accidents_par_an.append(a)

    v = pd.read_parquet(
        veh_path,
        columns=["Num_Acc", "id_vehicule", "catv"],
    )
    u = pd.read_parquet(
        usa_path,
        columns=["Num_Acc", "id_vehicule", "catu", "grav"],
    )

    for df in (v, u):
        df["Num_Acc"] = df["Num_Acc"].astype("string")
        df["id_vehicule"] = df["id_vehicule"].astype("string")

    v = v.loc[
        v["Num_Acc"].isin(a["Num_Acc"])
        & pd.to_numeric(v["catv"], errors="coerce").eq(CATV)
    ].copy()

    u = u.loc[
        u["Num_Acc"].isin(a["Num_Acc"])
        & pd.to_numeric(u["catu"], errors="coerce").isin([1, 2])
    ].copy()

    if v.duplicated(["Num_Acc", "id_vehicule"]).any():
        raise ValueError(
            f"{year} : véhicules de la catégorie dupliqués"
        )

    cible = u.merge(
        v[["Num_Acc", "id_vehicule"]],
        on=["Num_Acc", "id_vehicule"],
        how="inner",
        validate="many_to_one",
    )

    grav = pd.to_numeric(cible["grav"], errors="coerce")
    for code, nom in (
        (1, "indemnes"),
        (4, "blesses_legers"),
        (3, "hospitalises"),
        (2, "tues"),
    ):
        cible[nom] = grav.eq(code).astype(int)

    par_accident = (
        cible.groupby("Num_Acc", as_index=False)[
            ["indemnes", "blesses_legers", "hospitalises", "tues"]
        ]
        .sum()
    )
    par_accident["annee"] = year
    gravite_par_an.append(par_accident)

acc = pd.concat(accidents_par_an, ignore_index=True)
gravite_acc = pd.concat(gravite_par_an, ignore_index=True)

if acc.duplicated(["annee", "Num_Acc"]).any():
    raise ValueError("Accident dupliqué après concaténation")

for col in toutes_catv:
    if col not in acc:
        acc[col] = 0

acc = acc.merge(
    gravite_acc,
    on=["annee", "Num_Acc"],
    how="left",
    validate="one_to_one",
)

etats = ["indemnes", "blesses_legers", "hospitalises", "tues"]
for etat in etats:
    acc[etat] = acc[etat].fillna(0).astype(int)
    acc[f"avec_{etat}"] = acc[etat].gt(0)

acc["zone"] = (
    pd.to_numeric(acc["agg"], errors="coerce")
    .map({
        1: "Hors agglomération",
        2: "En agglomération",
    })
    .fillna("Zone inconnue")
)
acc["type_voie"] = (
    acc["highway"].astype("string").fillna("Non apparié")
)

# ── Nature des tiers : un seul groupe par accident ──────────────────

catv_numeriques = {
    col: int(col.removeprefix("n_catv_"))
    for col in toutes_catv
}

if CATV not in NON_MOTORISES and CATV in {0, 99}:
    raise ValueError(
        "La catégorie cible est indéterminée : impossible de "
        "définir correctement les tiers motorisés."
    )

colonnes_non_motorisees = [
    col for col, code in catv_numeriques.items()
    if code in NON_MOTORISES and code != CATV
]
colonnes_motorisees = [
    col for col, code in catv_numeriques.items()
    if code not in NON_MOTORISES
    and code not in {CATV, 0, 99}
]
colonnes_indeterminees = [
    col for col, code in catv_numeriques.items()
    if code in {0, 99}
]

def somme_lignes(colonnes):
    if not colonnes:
        return pd.Series(0, index=acc.index)
    return (
        acc[colonnes]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
    )

nb_cible = pd.to_numeric(acc[CATV_COL], errors="coerce").fillna(0)
nb_vehicules = pd.to_numeric(
    acc["n_vehicules"], errors="coerce"
)
nb_pietons = pd.to_numeric(
    acc["n_pietons"], errors="coerce"
).fillna(0)

autre_meme_categorie = nb_cible.gt(1)
autre_non_motorise = somme_lignes(
    colonnes_non_motorisees
).gt(0)
autre_motorise = somme_lignes(
    colonnes_motorisees
).gt(0)
autre_indetermine = (
    somme_lignes(colonnes_indeterminees).gt(0)
    | pd.to_numeric(
        acc["n_catv_inconnu"], errors="coerce"
    ).fillna(0).gt(0)
)
avec_pieton = nb_pietons.gt(0)

# Les cinq indicateurs décrivent les tiers, pas les victimes.
tiers_flags = pd.DataFrame(
    {
        "piéton": avec_pieton,
        "même catégorie": autre_meme_categorie,
        "non motorisé": autre_non_motorise,
        "motorisé": autre_motorise,
        "indéterminé": autre_indetermine,
    },
    index=acc.index,
)

n_types_tiers = tiers_flags.sum(axis=1)
autres_vehicules_connus = (
    nb_cible
    + somme_lignes(colonnes_non_motorisees)
    + somme_lignes(colonnes_motorisees)
    + somme_lignes(colonnes_indeterminees)
    + pd.to_numeric(
        acc["n_catv_inconnu"], errors="coerce"
    ).fillna(0)
)

incoherents = (
    nb_vehicules.isna()
    | autres_vehicules_connus.ne(nb_vehicules)
)
if incoherents.any():
    raise ValueError(
        f"{incoherents.sum()} accident(s) : la somme des "
        "n_catv_* ne correspond pas à n_vehicules. "
        "Classification des tiers interrompue."
    )

conditions = [
    n_types_tiers.eq(0),
    n_types_tiers.eq(1) & avec_pieton,
    n_types_tiers.eq(1) & autre_meme_categorie,
    n_types_tiers.eq(1) & autre_non_motorise,
    n_types_tiers.eq(1) & autre_motorise,
    n_types_tiers.eq(1) & autre_indetermine,
]

libelles_tiers = [
    "Solo (sans piéton)",
    "Piéton seul",
    f"Autre {LABEL} seulement",
    "Autre véhicule non motorisé seulement",
    "Autre véhicule motorisé seulement",
    "Véhicule indéterminé seulement",
]

acc["categorie_accident"] = "Combinaison de tiers"
for condition, libelle in zip(conditions, libelles_tiers):
    acc.loc[condition, "categorie_accident"] = libelle

# ── Résultat : zone × voie × catégorie d'accident ───────────────────

group_cols = ["zone", "type_voie", "categorie_accident"]
g = acc.groupby(group_cols, dropna=False)

n_col = f"Accidents {LABEL}"
table = g.size().rename(n_col).to_frame()

libelles_gravite = {
    "indemnes": "indemne",
    "blesses_legers": "blessé léger",
    "hospitalises": "hospitalisé",
    "tues": "tué",
}

for etat, libelle in libelles_gravite.items():
    col_accidents = f"Avec {LABEL} {libelle}"
    col_part = f"Part avec {LABEL} {libelle} (%)"
    col_personnes = f"{LABEL} {etat} (personnes)"

    table[col_accidents] = g[f"avec_{etat}"].sum().astype(int)
    table[col_part] = (
        100 * table[col_accidents] / table[n_col]
    ).round(1)
    table[col_personnes] = g[etat].sum().astype(int)

table = (
    table.reset_index()
    .sort_values(
        ["zone", "type_voie", n_col],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

table.to_csv(f"accidents_{LABEL}_detail_{YEARS[0]}_{YEARS[-1]+1}_global.csv")
table

,zone,type_voie,categorie_accident,Accidents CYCLO,Avec CYCLO indemne,Part avec CYCLO indemne (%),CYCLO indemnes (personnes),Avec CYCLO blessé léger,Part avec CYCLO blessé léger (%),CYCLO blesses_legers (personnes),Avec CYCLO hospitalisé,Part avec CYCLO hospitalisé (%),CYCLO hospitalises (personnes),Avec CYCLO tué,Part avec CYCLO tué (%),CYCLO tues (personnes)
0,En agglomération,Non apparié,Autre véhicule motorisé seulement,28,1,3.6,1,10,35.7,10,19,67.9,19,1,3.6,1
1,En agglomération,Non apparié,Solo (sans piéton),15,1,6.7,1,3,20.0,3,11,73.3,12,3,20.0,3
2,En agglomération,Non apparié,Autre CYCLO seulement,5,1,20.0,1,4,80.0,6,4,80.0,4,0,0.0,0
3,En agglomération,Non apparié,Piéton seul,3,2,66.7,2,0,0.0,0,0,0.0,0,0,0.0,0
4,En agglomération,Non apparié,Autre véhicule non motorisé seulement,2,0,0.0,0,1,50.0,1,1,50.0,1,0,0.0,0
5,En agglomération,cycleway,Autre véhicule motorisé seulement,125,10,8.0,10,93,74.4,100,26,20.8,27,1,0.8,1
6,En agglomération,cycleway,Solo (sans piéton),24,1,4.2,1,12,50.0,12,11,45.8,11,1,4.2,1
7,En agglomération,cycleway,Piéton seul,17,10,58.8,10,6,35.3,6,0,0.0,0,0,0.0,0
8,En agglomération,cycleway,Autre véhicule non motorisé seulement,14,10,71.4,12,3,21.4,3,1,7.1,1,0,0.0,0
9,En agglomération,cycleway,Véhicule indéterminé seulement,3,0,0.0,0,3,100.0,3,0,0.0,0,0,0.0,0
